# 038 — Regresión lineal, regularización y diagnóstico

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Modelo lineal:** `ŷ = β₀ + Σ βⱼxⱼ`. OLS minimiza `RSS = Σ(yᵢ−ŷᵢ)²`; solución cerrada
`β̂ = (XᵀX)⁻¹Xᵀy`. Para una feature: `β̂₁ = Σ(x−x̄)(y−ȳ)/Σ(x−x̄)²`, `β̂₀ = ȳ − β̂₁x̄`.
Calidad del ajuste: `R² = 1 − RSS/TSS` (solo dentro de muestra).

**Regularización** (con features escaladas, β₀ sin penalizar):

- **Ridge (L2):** `+ λΣβⱼ²` — encoge coeficientes, estabiliza colinealidad, no anula.
- **Lasso (L1):** `+ λΣ|βⱼ|` — puede anular coeficientes: selección de variables sparse.
- λ se elige en validación: acepta sesgo a cambio de menos varianza.

**Diagnóstico de residuos** `e = y − ŷ`: curva → falta no-linealidad; abanico →
heterocedasticidad; rachas → dependencia temporal; colas pesadas → outliers/pérdida robusta.
R² alto con residuos estructurados = modelo equivocado.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** x̄ = 2.5, ȳ = 5. Σ(x−x̄)(y−ȳ) = (−1.5)(−2) + (−0.5)(0) + (0.5)(−1) +
(1.5)(3) = 3 + 0 − 0.5 + 4.5 = 7. Σ(x−x̄)² = 2.25+0.25+0.25+2.25 = 5.
**β̂₁ = 1.4, β̂₀ = 5 − 1.4·2.5 = 1.5.** Predicciones [2.9, 4.3, 5.7, 7.1]; residuos
[0.1, 0.7, −1.7, 0.9]; RSS = 0.01+0.49+2.89+0.81 = 4.2; TSS = 4+0+1+9 = 14;
**R² = 1 − 4.2/14 = 0.7.**

**Ejercicio 2.** Con datos centrados, Σxᵢyᵢ = 7 y Σxᵢ² = 5, así que β̂₁(λ) = 7/(5+λ):
λ=0 → 1.40; λ=1 → 1.17; λ=5 → 0.70; λ=20 → 0.28. El coeficiente decrece monótonamente;
cuando λ → ∞, β̂₁ → 0 y el modelo predice siempre ȳ (la media): máxima regularización =
modelo constante = baseline.

**Ejercicio 3.** (a) Homocedasticidad: la varianza del error no es constante.
(b) R² compara contra la media global y puede seguir alto aunque los errores relativos en
casas caras sean enormes: mide varianza explicada, no estructura del residuo.
(c) Correcciones: modelar `log(y)` en lugar de y (estabiliza varianza multiplicativa) o
usar mínimos cuadrados ponderados; también revisar si falta una feature de segmento.

**Ejercicio 4.** El barrido de `candidates` con su accuracy de desarrollo es la búsqueda
de hiperparámetro; si los candidatos fueran λ ∈ {0, 0.1, 1, 10}, el flujo sería idéntico:
entrenar con train, comparar en validación y sellar test. El laboratorio muestra además el
sesgo optimista de elegir y medir sobre el mismo conjunto (declarado en `limitations`).


In [ ]:
result = run_lab("ml", seed=38)
assert result["kind"] == "ml"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 1 — OLS a mano, verificado
x = [1, 2, 3, 4]
y = [3, 5, 4, 8]
n = len(x)
x_bar, y_bar = sum(x) / n, sum(y) / n
beta1 = sum((xi - x_bar) * (yi - y_bar) for xi, yi in zip(x, y)) / sum((xi - x_bar) ** 2 for xi in x)
beta0 = y_bar - beta1 * x_bar
pred = [beta0 + beta1 * xi for xi in x]
res = [yi - pi for yi, pi in zip(y, pred)]
rss = sum(e ** 2 for e in res)
tss = sum((yi - y_bar) ** 2 for yi in y)
print(f"beta1={beta1:.2f} beta0={beta0:.2f} RSS={rss:.2f} TSS={tss:.2f} R2={1 - rss / tss:.3f}")


In [ ]:
# Ejercicio 2 — camino de encogimiento ridge
xc = [xi - x_bar for xi in x]
yc = [yi - y_bar for yi in y]
sxy = sum(a * b for a, b in zip(xc, yc))
sxx = sum(a * a for a in xc)
for lam in (0, 1, 5, 20):
    print(f"lambda={lam:>2}  beta1_ridge={sxy / (sxx + lam):.3f}")
# lambda→∞ ⇒ beta1→0 ⇒ el modelo predice la media: el baseline es el límite de la regularización


## Reflexión

1. El laboratorio selecciona un umbral discreto maximizando accuracy; una regresión
   minimiza una pérdida continua (RSS). ¿Qué gana y qué pierde cada formulación cuando el
   objetivo real es ordenar casos por riesgo en lugar de acertar una etiqueta?
2. Si duplicas una feature (columna copiada exacta) y ajustas OLS, ¿qué pasa con `XᵀX` y
   con los coeficientes? ¿Cómo resuelven el problema ridge y lasso, y de forma distinta?
3. Tu modelo lineal tiene R² = 0.95 en train y 0.58 en validación. ¿Qué dos causas
   distintas explicarías primero y qué gráfico de residuos pedirías para separarlas?
